# Sample 02: ビットフィールドとアライメント制御

ネットワークパケットや組み込み通信で頻出する、1バイト未満のビット単位パッキング (`Bits[N]`)、境界パディング (`align=N`)、および自然境界自動アライメント (`auto_align=True`) を学びます。

### 学べる内容
- `Bits[N]` によるビット単位のフィールド定義と `bits=16` などの総ビット幅コンテナ
- `align=4` による特定バイト境界へのアライメントパディング挿入
- `auto_align=True` によるメンバ型サイズに応じた自然アライメントの自動適用

In [1]:
from binary_master import (
    Bits,
    UInt8,
    UInt16,
    UInt32,
    binary_struct,
    hexdump,
    read_struct,
    sizeof,
)

## 1. 16ビットのビットフィールド構造体 (`Bits[N]`)

各フラグや数値をビット単位で詰め込みます。`bits=16` を指定することで、合計が16ビット（2バイト）であることを静的・動的に検証します。

In [2]:
@binary_struct(bits=16)
class DeviceStatus:
    """16ビットにパックされたハードウェア状態フラグ."""
    powered_on: Bits[1]   # Bit 0: 1 = Online, 0 = Standby
    busy: Bits[1]         # Bit 1: 1 = Processing, 0 = Idle
    mode: Bits[3]         # Bits 2..4: 動作モード (0..7)
    error_code: Bits[3]   # Bits 5..7: エラーコード (0..7)
    battery_pct: Bits[7]  # Bits 8..14: バッテリー残量 (0..100)
    reserved: Bits[1]     # Bit 15: 拡張用予約

print(f"DeviceStatus のサイズ: {sizeof(DeviceStatus)} バイト")

status = DeviceStatus(
    powered_on=1,
    busy=0,
    mode=5,
    error_code=2,
    battery_pct=95,
    reserved=0,
)
status_bytes = status.to_bytes()
print(f"シリアライズバイナリ: 0x{status_bytes.hex(' ')}")

restored_status = read_struct(DeviceStatus, status_bytes)
print(f"  powered_on:  {restored_status.powered_on}")
print(f"  busy:        {restored_status.busy}")
print(f"  mode:        {restored_status.mode}")
print(f"  error_code:  {restored_status.error_code}")
print(f"  battery_pct: {restored_status.battery_pct}%")

DeviceStatus のサイズ: 2 バイト
シリアライズバイナリ: 0x55 5f
  powered_on:  1
  busy:        0
  mode:        5
  error_code:  2
  battery_pct: 95%

## 2. 構造体のアライメント制御 (`align=4`)

C言語のコンパイラ配置と同様に、各フィールドの境界や末尾を4バイト境界に揃えるパディングを自動挿入します。
- `type_id` (1B) の後ろに 3B パディングが自動挿入され、`counter` (4B) はオフセット 4 から開始。
- `flags` (2B) がオフセット 8 に配置され、構造体全体のサイズを 4 の倍数にするため末尾に 2B パディングが挿入されて合計 12B になります。

In [3]:
@binary_struct(align=4)
class AlignedPacket:
    """4バイトアライメントを強制したパケット."""
    type_id: UInt8
    counter: UInt32
    flags: DeviceStatus

print(f"AlignedPacket のサイズ: {sizeof(AlignedPacket)} バイト (4の倍数)")
packet = AlignedPacket(type_id=0x01, counter=1000, flags=status)
packet_bytes = packet.to_bytes()
print(f"バイナリダンプ ({len(packet_bytes)} バイト):")
print(hexdump(packet_bytes, annotate=True))

restored_packet = read_struct(AlignedPacket, packet_bytes)
assert restored_packet.counter == 1000
assert restored_packet.flags.battery_pct == 95

AlignedPacket のサイズ: 12 バイト (4の倍数)
バイナリダンプ (12 バイト):
Offset    00 01 02 03 04 05 06 07  08 09 0A 0B 0C 0D 0E 0F  |     ASCII      |
------------------------------------------------------------------------------
00000000  01 00 00 00 e8 03 00 00  55 5f 00 00              |........U_..    |
  [Total: 12 bytes (`0x000C`)]

## 3. 自然境界自動アライメント (`auto_align=True`)

`auto_align=True` を指定すると、メンバの型サイズ（UInt16 は 2B 境界、UInt32 は 4B 境界）に合わせてパディングが自動挿入されます。

In [4]:
@binary_struct(auto_align=True)
class NaturalAlignedStruct:
    """各型の自然境界に合わせて自動パディングが挿入される構造体."""
    a: UInt8   # offset 0 (1B) -> 1B パディング挿入
    b: UInt16  # offset 2 (2B)
    c: UInt32  # offset 4 (4B)

print(f"NaturalAlignedStruct サイズ: {sizeof(NaturalAlignedStruct)} バイト")
natural = NaturalAlignedStruct(a=0xAA, b=0xBBCC, c=0x11223344)
natural_bytes = natural.to_bytes()
print(f"バイナリダンプ ({len(natural_bytes)} バイト):")
print(hexdump(natural_bytes, annotate=True))
print("アライメントとビットフィールドの検証成功！")

NaturalAlignedStruct サイズ: 8 バイト
バイナリダンプ (8 バイト):
Offset    00 01 02 03 04 05 06 07  08 09 0A 0B 0C 0D 0E 0F  |     ASCII      |
------------------------------------------------------------------------------
00000000  aa 00 cc bb 44 33 22 11                           |....D3".        |
  [Total: 8 bytes (`0x0008`)]
アライメントとビットフィールドの検証成功！